# MLP Matrix Learning 2
This notebook trains a multi-layer perception network using triplet margin loss to understand similar commits, using embeddings of commits as training data and input. This version uses more training data compared to the previous version.

# Obtain commit hashes and embedding

In [1]:
import os
import glob
import json

commit_hash_folder = os.path.join(os.getcwd(), "data/commit_hashes")
commit_hashes_paths = glob.glob(os.path.join(commit_hash_folder,'*.json'))

commit_hashes_paths

['/Users/savirarama/Documents/Embedding-Clustering-Visualization/data/commit_hashes/commit_hashes_calcite.json',
 '/Users/savirarama/Documents/Embedding-Clustering-Visualization/data/commit_hashes/commit_hashes_accumulo.json',
 '/Users/savirarama/Documents/Embedding-Clustering-Visualization/data/commit_hashes/commit_hashes_ambari.json']

### Concatenate lists of commit hashes

In [2]:
commit_hashes = []

for path in commit_hashes_paths:
    with open(path, 'r') as file:
        commit_hash = json.load(file)
    for hash in commit_hash:
        commit_hashes.append(hash)
        

In [3]:
len(commit_hashes)

40257

### Concatenate list of embeddings

In [4]:
import numpy as np

embedding_folder = os.path.join(os.getcwd(), "data/initial_embedding")
embedding_paths = glob.glob(os.path.join(embedding_folder, '*.npy'))

embeddings = np.load(embedding_paths[0])

for path in embedding_paths[1:]:
    embedding = np.load(path)
    embeddings = np.vstack((embeddings, embedding))

In [5]:
embeddings.shape

(40257, 768)

## Create training dataset

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class TripleCommitDataset(Dataset):
    def __init__(self, anchors, positive, negative):
        self.anchors = torch.FloatTensor(anchors)
        self.positive = torch.FloatTensor(positive)
        self.negative = torch.FloatTensor(negative)
        
        # Normalize the embeddings
        self.anchors = nn.functional.normalize(self.anchors, p=2, dim=1)
        self.positive = nn.functional.normalize(self.positive, p=2, dim=1)
        self.negative = nn.functional.normalize(self.negative, p=2, dim=1)

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        anchor = self.anchors[idx]
        pos = self.positive[idx]
        neg = self.negative[idx]
        return anchor, pos, neg
    

In [7]:
import json
triplets_dataset_path = 'data/triplets_accumulo_ambari_calcite.json'
with open(triplets_dataset_path, 'r') as file:
    triplets_data = json.load(file)

anchors = [entry['anchor'] for entry in triplets_data]
positives = [entry['positive'] for entry in triplets_data]
negatives = [entry['negative'] for entry in triplets_data]




In [8]:
anchors_embeddings = [embeddings[commit_hashes.index(anchor)] for anchor in anchors]
positives_embeddings = [embeddings[commit_hashes.index(pos)] for pos in positives]
negatives_embeddings = [embeddings[commit_hashes.index(neg)] for neg in negatives]

In [9]:
dataset = TripleCommitDataset(anchors_embeddings, positives_embeddings, negatives_embeddings)
dataloader = DataLoader(dataset, batch_size=3, shuffle=True)

/var/folders/hq/mr_d9f8x11dfzc_f03z9xzbw0000gn/T/ipykernel_44867/842328156.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  self.anchors = torch.FloatTensor(anchors)


## Define MLP

In [10]:
class MLPEmbedding(nn.Module):
    def __init__(self, input_dim=768, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
                    nn.Linear(768, 512),
                    nn.BatchNorm1d(512),
                    nn.ReLU(),
                    nn.Linear(512, 256),
                    nn.BatchNorm1d(256),
                    nn.ReLU(),
                    nn.Linear(256, 64)
                )
    def forward(self, x):
        return self.net(x)
    
model = MLPEmbedding(768, 64)

## Setup Optimiser and Loss

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
triplet_loss = nn.TripletMarginLoss(margin=1.0)

## Train the MLP

In [12]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for anchor, positive, negative in dataloader:
        optimizer.zero_grad()
        
        anchor_out = model(anchor)
        positive_out = model(positive)
        negative_out = model(negative)
        
        loss_value = triplet_loss(anchor_out, positive_out, negative_out)
        loss_value.backward()
        optimizer.step()
        
        total_loss += loss_value.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 512])